In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/code/src/")

In [11]:
from abc import ABC, abstractmethod
import subprocess
import json
from typing import Dict

In [ ]:
from data_processing.utils.geometry_utils import get_ned_rotation_from_yaw_pitch_roll, get_3d_point_distance, get_euler_diff, get_homogenous_matrix
from data_processing.utils.gps_utils import ned_from_gps, convert_ned_cam_to_opengl, ecef_from_gps
from data_processing.utils.metadata_utils import read_metadata_exiftool, read_video_metadata_exiftool

## Helper functions

In [22]:
def get_gps_value(metadata:Dict, tags_map:CameraTagsMap, absolute_altitude=True):
    latituide = float(metadata[tags_map.get_image_latitude_tag()])
    longitude = float(metadata[tags_map.get_image_longitude_tag()])
    if absolute_altitude:
        altitude = float(metadata[tags_map.get_image_abs_altitude_tag()])
    else:
        altitude = float(metadata[tags_map.get_image_rel_altitude_tag()])
    
    return latituide, longitude, altitude


def get_robot_rotation(metadata:Dict, tags_map:CameraTagsMap):
    flight_yaw = float(metadata[tags_map.get_robot_image_yaw_tag()])
    flight_Pitch = float(metadata[tags_map.get_robot_image_pitch_tag()])
    flight_roll = float(metadata[tags_map.get_robot_image_roll_tag()])

    R = get_ned_rotation_from_yaw_pitch_roll(flight_yaw, flight_Pitch, flight_roll)

    return R, (flight_yaw, flight_Pitch, flight_roll)

def get_camera_rotation(metadata:Dict, tags_map:CameraTagsMap):
    gimbal_yaw = float(metadata[tags_map.get_camera_image_yaw_tag()])
    gimbal_Pitch = float(metadata[tags_map.get_camera_image_pitch_tag()])
    gimbal_roll = float(metadata[tags_map.get_camera_image_roll_tag()])

    R = get_ned_rotation_from_yaw_pitch_roll(gimbal_yaw, gimbal_Pitch, gimbal_roll)

    return R, (gimbal_yaw, gimbal_Pitch, gimbal_roll)

In [ ]:
def write_camera_tags(metadata:Dict, image_file:str, tags_map:CameraTagsMap):
    cmd = ["exiftool", "-overwrite_original",
           f"-{tags_map.get_camera_image_yaw_tag()}={metadata[tags_map.get_camera_image_yaw_tag()]}",
           f"-{tags_map.get_camera_image_pitch_tag()}={metadata[tags_map.get_camera_image_pitch_tag()]}",
           f"-{tags_map.get_camera_image_roll_tag()}={metadata[tags_map.get_camera_image_roll_tag()]}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result
	

def write_gps_tags(metadata, image_file, tags_map:CameraTagsMap):
    cmd = ["exiftool", "-overwrite_original",
           f"-{tags_map.get_image_latitude_tag()}={metadata[tags_map.get_image_latitude_tag()]}",
           f"-{tags_map.get_image_longitude_tag()}={metadata[tags_map.get_image_longitude_tag()]}",
           f"-{tags_map.get_image_rel_altitude_tag()}={metadata[tags_map.get_image_rel_altitude_tag()]}",
           f"-{tags_map.get_image_abs_altitude_tag()}={metadata[tags_map.get_image_abs_altitude_tag()]}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result


def write_robot_tags(metadata, image_file, tags_map:CameraTagsMap):
    cmd = ["exiftool", "-overwrite_original",
           f"-{tags_map.get_robot_image_yaw_tag()}={metadata[tags_map.get_robot_image_yaw_tag()]}",
           f"-{tags_map.get_robot_image_pitch_tag()}={metadata[tags_map.get_robot_image_pitch_tag()]}",
           f"-{tags_map.get_robot_image_roll_tag()}={metadata[tags_map.get_robot_image_roll_tag()]}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result


def write_standard_gps_tags(metadata, image_file, tags_map:CameraTagsMap, gps_tags_map:GpsTagsMap):
    latitudeRef = 'N' if metadata[tags_map.get_image_latitude_tag()] > 0 else 'S'
    longitudeRef = 'E' if metadata[tags_map.get_image_longitude_tag()] > 0 else 'W'
    altitudeRef = 0 if metadata[tags_map.get_image_abs_altitude_tag()] > 0 else 1

    cmd = ["exiftool", "-overwrite_original",
           f"-{gps_tags_map.get_ref_latitude_tag()}={latitudeRef}",
           f"-{gps_tags_map.get_abs_latitude_tag()}={abs(metadata[tags_map.get_image_latitude_tag()])}",
           f"-{gps_tags_map.get_ref_longitude_tag()}={longitudeRef}",
           f"-{gps_tags_map.get_abs_longitude_tag()}={abs(metadata[tags_map.get_image_longitude_tag()])}",
           f"-{gps_tags_map.get_ref_altitude_tag()}={altitudeRef}",
           f"-{gps_tags_map.get_abs_altitude_tag()}={abs(metadata[tags_map.get_image_abs_altitude_tag()])}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result
	
	
def get_distance_between_camera_centers(camera1_metadata:Dict, camera2_metadata:Dict, tags_map:CameraTagsMap):
    # first we get the Abslout XYZ camera center value to measure the distance
    X1, Y1, Z1 = ecef_from_gps(lat_deg=camera1_metadata[tags_map.get_image_latitude_tag()],
                               long_deg=camera1_metadata[tags_map.get_image_longitude_tag()],
                               alt_m=camera1_metadata[tags_map.get_image_abs_altitude_tag()])
    
    X2, Y2, Z2 = ecef_from_gps(lat_deg=camera2_metadata[tags_map.get_image_latitude_tag()],
                               long_deg=camera2_metadata[tags_map.get_image_longitude_tag()],
                               alt_m=camera2_metadata[tags_map.get_image_abs_altitude_tag()])
    
    d = get_3d_point_distance([X1, Y1, Z1], [X2, Y2, Z2])

    return d


def get_euler_diff_between_cameras(camera1_metadata:Dict, camera2_metadata:Dict, tags_map:CameraTagsMap):
    # we will get the shortest path using the formula (diff + 180) % 360 - 180
    
    yaw_diff = get_euler_diff(camera2_metadata[tags_map.get_camera_image_yaw_tag()], camera1_metadata[tags_map.get_camera_image_yaw_tag()])
    pitch_diff = get_euler_diff(camera2_metadata[tags_map.get_camera_image_pitch_tag()], camera1_metadata[tags_map.get_camera_image_pitch_tag()])
    roll_diff = get_euler_diff(camera2_metadata[tags_map.get_camera_image_roll_tag()], camera1_metadata[tags_map.get_camera_image_roll_tag()])

    return yaw_diff, pitch_diff, roll_diff

In [ ]:
def get_frame_metadata(video_metadata:Dict, frame_number:int, camera_tags_map:CameraTagsMap, video_tags_map:CameraTagsMap):
    
    frame_data = {camera_tags_map.get_image_latitude_tag(): video_metadata.get(video_tags_map.get_frame_latitude_tag(), None),
                  camera_tags_map.get_image_longitude_tag(): video_metadata.get(video_tags_map.get_frame_longitude_tag(), None),
                  camera_tags_map.get_image_abs_altitude_tag(): video_metadata.get(video_tags_map.get_frame_abs_altitude_tag(), None),
                  camera_tags_map.get_image_rel_altitude_tag(): video_metadata.get(video_tags_map.get_frame_rel_altitude_tag(), None),
                  camera_tags_map.get_camera_image_yaw_tag(): video_metadata.get(video_tags_map.get_camera_frame_yaw_tag(), None),
                  camera_tags_map.get_camera_image_pitch_tag(): video_metadata.get(video_tags_map.get_camera_frame_pitch_tag(), None),
                  camera_tags_map.get_camera_image_roll_tag(): video_metadata.get(video_tags_map.get_camera_frame_roll_tag(), None),
                  camera_tags_map.get_robot_image_yaw_tag(): video_metadata.get(video_tags_map.get_robot_frame_yaw_tag(), None),
                  camera_tags_map.get_robot_image_pitch_tag(): video_metadata.get(video_tags_map.get_robot_frame_pitch_tag(), None),
                  camera_tags_map.get_robot_image_roll_tag(): video_metadata.get(video_tags_map.get_robot_frame_roll_tag(), None)}
    return frame_data

## Classes

In [12]:
class CameraTagsMap(ABC):
    def __init__(self):
        pass
    def get_image_longitude_tag(self):
        pass
    def get_image_latitude_tag(self):
        pass
    def get_image_rel_altitude_tag(self):
        pass
    def get_image_abs_altitude_tag(self):
        pass
    def get_frame_longitude_tag(self, idx):
        pass
    def get_frame_latitude_tag(self, idx):
        pass
    def get_frame_abs_altitude_tag(self, idx):
        pass
    def get_frame_rel_altitude_tag(self, idx):
        pass

    def get_camera_image_yaw_tag(self):
        pass
    def get_camera_image_pitch_tag(self):
        pass
    def get_camera_image_roll_tag(self):
        pass
    def get_camera_frame_yaw_tag(self, idx):
        pass
    def get_camera_frame_pitch_tag(self, idx):
        pass
    def get_camera_frame_roll_tag(self, idx):
        pass

    def get_robot_image_yaw_tag(self):
        pass
    def get_robot_image_pitch_tag(self):
        pass
    def get_robot_image_roll_tag(self):
        pass
    def get_robot_frame_yaw_tag(self, idx):
        pass
    def get_robot_frame_pitch_tag(self, idx):
        pass
    def get_robot_frame_roll_tag(self, idx):
        pass

In [ ]:
class GpsTagsMap():
    def __init__(self):
        self.tags_map = {
            "longitude_abs":"GPS:GPSLongitude",
            "longitude_ref":"GPS:GPSLongitudeRef",
            "latitude_abs":"GPS:GPSLatitude",
            "latitude_ref":"GPS:GPSLatitudeRef",
            "altitude_abs":"GPS:GPSAltitude",
            "altitude_ref":"GPS:GPSAltitudeRef",
        }
    
    def get_abs_longitude_tag(self):
        return self.tags_map["longitude_abs"]
    
    def get_ref_longitude_tag(self):
        return self.tags_map["longitude_ref"]
    
    
    def get_abs_latitude_tag(self):
        return self.tags_map["latitude_abs"]
    
    def get_ref_latitude_tag(self):
        return self.tags_map["latitude_ref"]
    

    def get_abs_altitude_tag(self):
        return self.tags_map["altitude_abs"]
    
    def get_ref_altitude_tag(self):
        return self.tags_map["altitude_ref"]

In [15]:
class CameraMetaData(ABC):
    def __init__(self, 
                 frame_file:str, 
                 camera_tags_map:CameraTagsMap,
                 gps_tags_map:GpsTagsMap,
                 absolute_altitude:bool=True):
        pass
    def read_gps_value(self):
        pass
    def set_ref_gps(self, longitude, latitude, altitude):
        pass
    def get_camera_rotation(self):
        pass
    def get_c2w_opengl(self):
        pass
    def write_camera_gps_tags(self):
        pass
    def write_standard_gps_tags(self):
        pass
    def write_all_camera_specific_tags(self):
        pass
    def write_camera_tags(self):
        pass
    def get_distance_between_camera_centers(self, camera1_metadata, camera2_metadata):
        pass
    def get_euler_diff_between_cameras(self, camera1_metadata, camera2_metadata):
        pass

In [16]:
class CameraVideoMetaData(ABC):
    def __init__(self, 
                 video_file, 
                 camera_tags_map,
                 absolute_altitude=True):
        pass
    def get_frame_metadata(self, idx):
        pass

In [17]:
class DjiDroneMini4TagsMap(CameraTagsMap):
    def __init__(self):
        self.tags_map = {
            "image_longitude":"XMP-drone-dji:GPSLongitude",
            "image_latitude":"XMP-drone-dji:GPSLatitude",
            "image_altitude":"XMP-drone-dji:RelativeAltitude",
            "image_altitude_abs":"XMP-drone-dji:AbsoluteAltitude",

            "frame_longitude":"GPSLongitude",
            "frame_latitude":"GPSLatitude",
            "frame_altitude ":"RelativeAltitude",
            "frame_altitude_abs ":"AbsoluteAltitude",

            "camera_image_yaw":"XMP-drone-dji:GimbalYawDegree",
            "camera_image_pitch":"XMP-drone-dji:GimbalPitchDegree",
            "camera_image_roll":"XMP-drone-dji:GimbalRollDegree",

            "camera_frame_yaw":"GimbalYaw",
            "camera_frame_pitch":"GimbalPitch",
            "camera_frame_roll":"GimbalRoll",

            "robot_image_yaw":"XMP-drone-dji:FlightYawDegree",
            "robot_image_pitch":"XMP-drone-dji:FlightPitchDegree",
            "robot_image_roll":"XMP-drone-dji:FlightRollDegree",

            "robot_frame_yaw":"DroneYaw",
            "robot_frame_pitch":"DronePitch",
            "robot_frame_roll":"DroneRoll",
        }

    def get_image_longitude_tag(self):
        return self.tags_map['image_longitude']
    
    def get_image_latitude_tag(self):
        return self.tags_map['image_latitude']
    
    def get_image_rel_altitude_tag(self):
        return self.tags_map['image_altitude']
    
    def get_image_abs_altitude_tag(self):
        return self.tags_map['image_altitude_abs']
    
    def get_frame_longitude_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['frame_longitude']}"
    
    def get_frame_latitude_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['frame_latitude']}"
    
    def get_frame_abs_altitude_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['frame_altitude']}"
    
    def get_frame_rel_altitude_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['frame_altitude_abs']}"

    def get_camera_image_yaw_tag(self):
        return self.tags_map['camera_image_yaw']
    
    def get_camera_image_pitch_tag(self):
        return self.tags_map['camera_image_pitch']
    
    def get_camera_image_roll_tag(self):
        return self.tags_map['camera_image_roll']
    
    def get_camera_frame_yaw_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['camera_frame_yaw']}"
    
    def get_camera_frame_pitch_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['camera_frame_pitch']}"
    
    def get_camera_frame_roll_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['camera_frame_roll']}"

    def get_robot_image_yaw_tag(self):
        return self.tags_map['robot_image_yaw']
    
    def get_robot_image_pitch_tag(self):
        return self.tags_map['robot_image_pitch']
    
    def get_robot_image_roll_tag(self):
        return self.tags_map['robot_image_roll']
    
    def get_robot_frame_yaw_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['robot_frame_yaw']}"
    
    def get_robot_frame_pitch_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['robot_frame_pitch']}"
    
    def get_robot_frame_roll_tag(self, idx):
        return f"Doc{str(idx)}:{self.tags_map['robot_frame_roll']}"

In [ ]:
class DjiDroneMini4ImageMetaData(CameraMetaData):
    def __init__(self, 
                 image_file:str, 
                 camera_tags_map:CameraTagsMap,
                 gps_tags_map:GpsTagsMap,
                 absolute_altitude:bool=True):
        self.image_file = image_file
        self.camera_tags_map = camera_tags_map
        self.gps_tags_map = gps_tags_map
        self.absolute_altitude = absolute_altitude
        self.metadata = read_metadata_exiftool(image_file)

        self.lat_0 = 0
        self.long_0 = 0
        self.alt_0 = 0
        
    def get_gps_value(self):
        return get_gps_value(self.metadata, self.camera_tags_map, self.absolute_altitude)
    
    def set_ref_gps(self, longitude, latitude, altitude):
        self.lat_0 = latitude
        self.long_0 = longitude
        self.alt_0 = altitude

    def get_camera_rotation(self):
        return get_camera_rotation(self.metadata, self.camera_tags_map)
    
    def get_c2w_opengl(self):
        lat, longitude, alt = self.get_gps_value()
        x, y, z = ned_from_gps(lat, longitude, alt, 
                               self.lat_0, self.long_0, self.alt_0, 
                               use_absolute_altitude=self.absolute_altitude)

        Rg2f, _ = self.get_camera_rotation()

        pose_g2w = get_homogenous_matrix(Rg2f, x, y, z)
        pose_c2w = convert_ned_cam_to_opengl(pose_g2w)

        return pose_c2w


    def write_camera_gps_tags(self):
        write_gps_tags(self.metadata, self.image_file, self.camera_tags_map)

    def write_standard_gps_tags(self):
        write_standard_gps_tags(self.metadata, self.image_file, 
                                tags_map=self.camera_tags_map, 
                                gps_tags_map=self.gps_tags_map)

    def write_camera_tags(self):
        write_camera_tags(self.metadata, self.image_file, self.camera_tags_map)

    def write_robot_tags(self):
        write_robot_tags(self.metadata, self.image_file, self.camera_tags_map)

    def write_all_camera_specific_tags(self):
        self.write_camera_gps_tags()
        self.write_camera_tags()
        self.write_robot_tags()

    def get_distance_between_camera_centers(self, camera1_metadata, camera2_metadata):
        get_distance_between_camera_centers(camera1_metadata, camera2_metadata, self.camera_tags_map)
    
    def get_euler_diff_between_cameras(self, camera1_metadata, camera2_metadata):
        get_euler_diff_between_cameras(camera1_metadata, camera2_metadata, self.camera_tags_map)

In [25]:
class DjiDroneMini4VideoMetaData(CameraVideoMetaData):
    def __init__(self, 
                 video_file, 
                 camera_tags_map,
                 absolute_altitude=True):
        self.metadata = read_video_metadata_exiftool(video_file)
        self.camera_tags_map = camera_tags_map
        self.absolute_altitude = absolute_altitude
        
    def get_frame_metadata(self, idx):
        return get_frame_metadata(self.metadata, frame_number=idx, camera_tags_map=self.camera_tags_map, video_tags_map=self.camera_tags_map)